In [1]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [2]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [3]:
tablaPromotion = "SpecialOffer"
esquemaSales = "Sales"

dimensionPromotion = pd.read_sql_table(tablaPromotion, motorBaseDatos, esquemaSales)
# dimensionPromotion.head()
dimensionPromotion

,SpecialOfferID,Description,DiscountPct,Type,Category,StartDate,EndDate,MinQty,MaxQty,rowguid,ModifiedDate
0,1,No Discount,0.00,No Discount,No Discount,2011-05-01,2014-11-30,0,NaN,0290c4f5-191f-4337-ab6b-0a2dde03cbf9,2011-04-01
1,2,Volume Discount 11 to 14,0.02,Volume Discount,Reseller,2011-05-31,2014-05-30,11,14.0,d7542ee7-15db-4541-985c-5cc27aef26d6,2011-05-01
2,3,Volume Discount 15 to 24,0.05,Volume Discount,Reseller,2011-05-31,2014-05-30,15,24.0,4bdbcc01-8cf7-40a9-b643-40ec5b717491,2011-05-01
3,4,Volume Discount 25 to 40,0.10,Volume Discount,Reseller,2011-05-31,2014-05-30,25,40.0,504b5e85-8f3f-4ebc-9e1d-c1bc5dea9aa8,2011-05-01
4,5,Volume Discount 41 to 60,0.15,Volume Discount,Reseller,2011-05-31,2014-05-30,41,60.0,677e1d9d-944f-4e81-90e8-47eb0a82d48c,2011-05-01
5,6,Volume Discount over 60,0.20,Volume Discount,Reseller,2011-05-31,2014-05-30,61,NaN,8157f569-4e8d-46b6-9347-5d0f726a9439,2011-05-01
6,7,Mountain-100 Clearance Sale,0.35,Discontinued Product,Reseller,2012-04-13,2012-05-29,0,NaN,7df15bf5-6c05-47e7-80a4-22bd1ce59a72,2012-03-14
7,8,Sport Helmet Discount-2002,0.10,Seasonal Discount,Reseller,2012-05-30,2012-06-29,0,NaN,20c5d2cc-a38f-48f8-ac9a-8f15943e52ae,2012-04-30
8,9,Road-650 Overstock,0.30,Excess Inventory,Reseller,2012-05-30,2012-07-30,0,NaN,0cf8472b-f9e6-4945-9e09-549d7dde2198,2012-04-30
9,10,Mountain Tire Sale,0.50,Excess Inventory,Customer,2013-05-14,2013-07-29,0,NaN,220444ad-2ef3-4e4c-87e9-3aa6ee39a877,2013-04-14


TRANSFORMACION

In [4]:
dimensionPromotion.rename(columns={
    'SpecialOfferID': 'PromotionKey',
    'Description' :'EnlishPromotionName',
    'Type' : 'EnglishPromotionType',
    'Category' : 'EnglishPromotionCategory'
},inplace=True)


dimensionPromotion["PromotionAlternateKey"] = dimensionPromotion["PromotionKey"]


dimensionPromotion["SpanishPromotionName"] = None

dimensionPromotion["FrenchPromotionName"] = None


dimensionPromotion["SpanishPromotionType"] = None

dimensionPromotion["FrenchPromotionType"] = None


dimensionPromotion["SpanishPromotionCategory"] = None

dimensionPromotion["FrenchPromotionCategory"] = None

dimensionPromotion.drop(columns=[
    'rowguid',
    'ModifiedDate'

] ,inplace=True)

# dimensionPromotion.head()
dimensionPromotion

,PromotionKey,EnlishPromotionName,DiscountPct,EnglishPromotionType,EnglishPromotionCategory,StartDate,EndDate,MinQty,MaxQty,PromotionAlternateKey,SpanishPromotionName,FrenchPromotionName,SpanishPromotionType,FrenchPromotionType,SpanishPromotionCategory,FrenchPromotionCategory
0,1,No Discount,0.00,No Discount,No Discount,2011-05-01,2014-11-30,0,NaN,1,None,None,None,None,None,None
1,2,Volume Discount 11 to 14,0.02,Volume Discount,Reseller,2011-05-31,2014-05-30,11,14.0,2,None,None,None,None,None,None
2,3,Volume Discount 15 to 24,0.05,Volume Discount,Reseller,2011-05-31,2014-05-30,15,24.0,3,None,None,None,None,None,None
3,4,Volume Discount 25 to 40,0.10,Volume Discount,Reseller,2011-05-31,2014-05-30,25,40.0,4,None,None,None,None,None,None
4,5,Volume Discount 41 to 60,0.15,Volume Discount,Reseller,2011-05-31,2014-05-30,41,60.0,5,None,None,None,None,None,None
5,6,Volume Discount over 60,0.20,Volume Discount,Reseller,2011-05-31,2014-05-30,61,NaN,6,None,None,None,None,None,None
6,7,Mountain-100 Clearance Sale,0.35,Discontinued Product,Reseller,2012-04-13,2012-05-29,0,NaN,7,None,None,None,None,None,None
7,8,Sport Helmet Discount-2002,0.10,Seasonal Discount,Reseller,2012-05-30,2012-06-29,0,NaN,8,None,None,None,None,None,None
8,9,Road-650 Overstock,0.30,Excess Inventory,Reseller,2012-05-30,2012-07-30,0,NaN,9,None,None,None,None,None,None
9,10,Mountain Tire Sale,0.50,Excess Inventory,Customer,2013-05-14,2013-07-29,0,NaN,10,None,None,None,None,None,None


CARGAR A LA BODEGA

In [5]:
dimensionPromotion.to_sql('dimensionPromotion',motorBodegaDatos, if_exists='replace',index=False)

16